# NB_Purview_Copilot_SilverToGold
**Source:** `AI_Medallion.Silver.purview_copilot_usage`  
**Target:** `AI_Medallion.Gold.fact_copilot_active`  
**Grain:** `GROUP BY` Okta user, activity date, interaction type — `Request` = `SUM(prompt_count)`  
**Filter:** groups with `SUM(prompt_count) >= 1` (all license types)  
**Load:** Incremental merge on `RecordSurrogateKey`  
**Run after:** `NB_Purview_Copilot_BronzeToSilver`

In [ ]:
import os
from pathlib import Path
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
os.environ.setdefault("MEDALLION_LOCAL_TABLE_CACHE", "1")
os.environ.setdefault("MEDALLION_SCAN_CHUNK_ROWS", "75000")
os.environ.setdefault("MEDALLION_SCAN_RETRIES", "6")


In [ ]:
%run ./Common/bootstrap


In [ ]:
silver = read_table_broken_lineage(TABLE_PURVIEW_COPILOT_SILVER, "purview_copilot_silver")

use_case_multiplier = F.lit(COPILOT_USE_CASE_MULTIPLIER).cast("decimal(18,8)")
automation_poc = F.lit(COPILOT_AUTOMATION_POC).cast("decimal(18,8)")
total_hours_saved_poc = F.lit(COPILOT_TOTAL_HOURS_SAVED_POC).cast("decimal(18,8)")


In [ ]:
apply_local_merge_spark_conf(spark)

grouped = break_lineage_local(
    purview_active_copilot_aggregate(silver), "purview_active_copilot_daily"
)
grouped = grouped.withColumn(
    FACT_PURVIEW_ACTIVE_COPILOT_MERGE_KEY, purview_active_copilot_merge_key()
)

interaction = F.col("purview_copilot_usage_interaction_type")

fact = grouped.select(
    F.col("Month_No"),
    F.col("Request"),
    F.lit(COPILOT_ASSET_ID).alias("assetId"),
    F.lit(COPILOT_ASSET_NAME).alias("assetName"),
    F.lit(None).cast("string").alias("eventType"),
    F.lit(None).cast("string").alias("id_project"),
    F.lit(None).cast("string").alias("project"),
    F.lit(None).cast("string").alias("account"),
    F.col("okta_user_id").alias("userId"),
    F.col("okta_user_full_name").alias("username"),
    F.col("okta_user_employee_number").alias("employeeid"),
    F.col("okta_user_email").alias("userEmail"),
    F.col("okta_user_full_email").alias("userFullEmail"),
    F.col("okta_user_region").alias("Division"),
    F.col("okta_user_country").alias("userCountry"),
    purview_copilot_use_case_code(interaction).alias("useCaseCode"),
    interaction.alias("useCaseDisplayName"),
    (
        F.col("Request").cast("decimal(18,8)") * total_hours_saved_poc * use_case_multiplier
    ).alias("UseCaseHoursSaved"),
    (
        F.col("Request").cast("decimal(18,8)") * (total_hours_saved_poc / automation_poc)
    ).alias("UseCaseTotalHours"),
    F.lit(1).cast("int").alias("UserWithActivity"),
    F.col("purview_copilot_usage_date").alias("LastModification"),
    F.col("hc_business_unit").alias("User_UN"),
    F.col("hc_job_title").alias("JobRoles"),
    F.col("hc_okta_user_country").alias("Employee_Country"),
    F.col("hc_office").alias("Employee_Office"),
    F.col("hc_talent_segment").alias("Segment"),
    F.col("hc_division").alias("Foundation"),
    F.col("hc_business_unit_code").alias("Employee_Business_Unit_Code"),
    F.lit(None).cast("string").alias("asset_model"),
    F.lit(None).cast("string").alias("asset_useCase"),
    F.lit(None).cast("string").alias("asset_mySourceType"),
    F.lit(None).cast("string").alias("asset_sourceType"),
    F.lit(None).cast("string").alias("billingClientName"),
    F.lit(None).cast("string").alias("billingCountry"),
    F.lit(None).cast("string").alias("billingRegion"),
    F.lit(None).cast("string").alias("flowid"),
    use_case_multiplier.alias("useCaseMultiplier"),
    automation_poc.alias("automation_poc"),
    total_hours_saved_poc.alias("totalHoursSaved_poc"),
    F.col(FACT_PURVIEW_ACTIVE_COPILOT_MERGE_KEY),
)

fact = fact.filter(
    F.col("Month_No").isNotNull()
    & F.col("Request").isNotNull()
    & F.col("assetId").isNotNull()
    & F.col("assetName").isNotNull()
    & F.col("useCaseCode").isNotNull()
    & F.col("useCaseDisplayName").isNotNull()
    & F.col("UseCaseHoursSaved").isNotNull()
    & F.col("UseCaseTotalHours").isNotNull()
    & F.col("UserWithActivity").isNotNull()
    & F.col("LastModification").isNotNull()
    & F.col("useCaseMultiplier").isNotNull()
    & F.col("automation_poc").isNotNull()
    & F.col("totalHoursSaved_poc").isNotNull()
    & F.col(FACT_PURVIEW_ACTIVE_COPILOT_MERGE_KEY).isNotNull()
)

publish_merge_staging(fact, TABLE_FACT_COPILOT_ACTIVE)


In [ ]:
merge_incremental(
    spark,
    fact,
    TABLE_FACT_COPILOT_ACTIVE,
    FACT_PURVIEW_ACTIVE_COPILOT_MERGE_KEY,
)
show_sample(fact.limit(10))
